In [1]:
import sys
sys.path.append("FINAL_SWH_OSV/python")
from __future__ import annotations
import json
import pandas as pd
import matplotlib.pyplot as plt
from cvss import CVSS3
import pyzstd 
from itertools import islice

import pandas as pd
from datetime import datetime
from dataclasses import dataclass
from typing import List
from dataclasses import asdict

from osv_preprocess.osv_range import OsvRange


from metadata_miner.Repo_metadata_store import RepoMetadataStore
from metadata_miner.head_metadata_store import HeadMetadataStore
from metadata_miner.range_metadata_store import RangeMetadataStore


from metadata_miner.archived import ArchivedMetadata
from metadata_miner.main_branch_metadata import MainBranchMetadata
from metadata_miner.divergence_metadata import DivergenceMetadata
from metadata_miner.cherry_pick import CherryPickMetadata

from osv_preprocess.osv_range import EventType

import multiprocessing as mp
from enum import Enum

current_folder='./FINAL_SWH_OSV/'
result_path = f'{current_folder}data/'
metadata_store_path = f"{result_path}metadata_store/"
#Path to the folder that will store the cloned repositories
workspace = f"{current_folder}cloned_repos/"

plt.style.use('ggplot') 
plt.rcParams['figure.facecolor'] = 'white'
SMALL_SIZE = 14
MEDIUM_SIZE = 16
BIGGER_SIZE = 18

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

CWD: /home/creux/workspace


In [2]:
manual_data_quality = f"{result_path}RQ_SECTION_4_VETTING_ANALYSIS/data_quality.csv"
metrics_5_path= f"{result_path}Metric5.jsonl.zst"
metadata_path= f"{result_path}origin-metadata.csv.zst"
analysis_out_path=f"{result_path}/analysis/"
#create output directory if it does not exist
import os
if not os.path.exists(analysis_out_path):
    os.makedirs(analysis_out_path)

In [3]:
class RepoMetadata:
    def __init__(self, origin:str,url:str,isFork:bool,stars:int,forks:int):
        self.origin=origin
        self.url=url
        self.isFork=isFork
        self.stars=stars
        self.forks=forks

    def to_string(self):
        return f"{self.origin},{self.url},{self.isFork},{self.stars},{self.forks}"

class ReposMetadata:
    reposMaps:dict[str,RepoMetadata]
    metadata_path:str
    skipped_repo : list[str] = []
    output_path:str = f"{analysis_out_path}reposMetadata_final.json"
    def __init__(self,metadata_path:str):
        self.reposMaps={}
        self.metadata_path=metadata_path
    
    def load_metadata_from_zst_csv(self,url_to_fetch:set[str]):
        self.reposMaps={}
        with pyzstd.open(metadata_path, "rb") as f:
            for line in f:
                line = line.decode("utf-8")
                line = line.split(",")
                url=line[1].replace('"""','')
                if url in url_to_fetch:
                    self.reposMaps[url]=RepoMetadata(line[0],url,line[2]=="True",int(line[3]),int(line[4]))
                else :
                    self.skipped_repo.append(url)
        print(f"Found {len(self.reposMaps)} skipp {len(self.skipped_repo)}")

    def export_to_json(self):
        with open(self.output_path, 'w') as f:
            json.dump({k: v.__dict__ for k, v in self.reposMaps.items()}, f, indent=4)
    
    def load_from_json()->ReposMetadata:
        with open(ReposMetadata.output_path, 'r') as f:
            data = json.load(f)
            result= ReposMetadata("")
            result.reposMaps = {k: RepoMetadata(**v) for k, v in data.items()}
        print(f"Loaded {len(result.reposMaps)} repos ")
        return result



### Forks with unpatched head 

In [ ]:
# Get the list of all the repositories, to fetch their metadata

url=set()
with pyzstd.open(metrics_5_path, 'r') as file:
            for line in file:
                        obj = json.loads(line)
                        forks_urls = obj['urls']
                        url.update(forks_urls)
                        range_url=obj['range']['repo_url']
                        url.add(range_url)
print(f"Need to fetch metadata for {len(url)} repositories")

reposMetadata = ReposMetadata(metadata_path)
reposMetadata.load_metadata_from_zst_csv(url)
reposMetadata.export_to_json()


In [ ]:
#Load from checkpoint
reposMetadata = ReposMetadata.load_from_json()


Loaded 1456038 repos 


In [5]:
@dataclass
class UnpatchedFork:
    unpatched_heads: List[str]
    url: str
    range: OsvRange
    timestamps: List[float]

    def to_dict(self):
        return {
            "unpatched_heads": self.unpatched_heads,
            "url": self.url,
            "range": self.range.__dict__,
            "timestamps": self.timestamps,
        }
    
    def get_latest_head(self)->str:
        latest_timestamp = self.timestamps[0]
        latest_head = self.unpatched_heads[0]
        for head, timestamp in zip(self.unpatched_heads, self.timestamps):
            if timestamp > latest_timestamp:
                latest_head = head
                latest_timestamp = timestamp
        return (latest_head,latest_timestamp)
            
class Analysis:
    def __init__(self,input_path:str,output_path:str):
        self.input_path=input_path
        self.output_path=output_path
        self.result :List[UnpatchedFork]= []
       
    def export_as_csv(self,csv_path:str)->"Analysis":
        dict={}
        for res in self.result:
            fork = res.url
            upstream = res.range.repo_url
            vuln = res.range.vulnerability_id
            latest_head = res.get_latest_head()
            latest_head = (latest_head[0].replace("swh:1:rev:","https://archive.softwareheritage.org/browse/revision/"),latest_head[1])
            #filter fix urls
            fix_urls = set(["https://archive.softwareheritage.org/browse/revision/"+id+"/" for id,event  in res.range.events.items() if event == EventType.FIXED])
            current_tuple= (fork,upstream,vuln," ".join(fix_urls))
            if current_tuple not in dict:
                dict[current_tuple]=latest_head
            current_head=dict[current_tuple]
            if latest_head[1] > current_head[1]:
                dict[current_tuple]=latest_head

        #Export to csv
        with open(csv_path, 'w') as f:
            f.write("fork,upstream,vuln,head_swh,fix_urls\n")
            for entry,latest_head in dict.items():
                f.write(f"{entry[0]},{entry[1]},{entry[2]},{latest_head[0]},{entry[3]}\n")
        return self
        


    
    def get_ranges_input(self) :
        range_set  = set()
        with pyzstd.open(self.input_path, 'r') as file:
                for line in file:
                        raw = json.loads(line)
                        range_events : dict[str, EventType] = dict(map(lambda x: (x[0], EventType(x[1])), raw["range"]['events'].items()))
                        range_obj = OsvRange(
                                vulnerability_id=raw["range"].get("vulnerability_id"),
                                repo_url=raw["range"].get("repo_url"),
                                severity=raw["range"].get("severity"),
                                events=range_events,

                                )
                        range_set.add(range_obj)
                               
                                
        print(f"Found {len(range_set)} unique ranges")
        
        return list(range_set)
    
    def get_ranges_input_par(self) -> "Analysis":
            range_set  = set()

            def line_generator():
                with pyzstd.open(self.input_path, 'r') as file:
                    for line in file:
                        yield line.strip()
            
            with mp.Pool() as pool:
                # Process lines as they're read, without storing all in memory
                for result in pool.imap(Analysis.parse_json_line, line_generator(), chunksize=50):
                    range_set.add(result.range)
            
            print(f"Found {len(range_set)} unique ranges")
            return list(range_set)
    
    def obj_to_UnpatchedFork(raw)->UnpatchedFork:
                        range_events : dict[str, EventType] = dict(map(lambda x: (x[0], EventType(x[1])), raw["range"]['events'].items()))
                        range_obj = OsvRange(
                            vulnerability_id=raw["range"].get("vulnerability_id"),
                            repo_url=raw["range"].get("repo_url"),
                            severity=raw["range"].get("severity"),
                            events=range_events,

                        )
                        #Handle cases where fork urls are in a list
                        unpatched_entries=[]
                        if 'urls' in raw :
                             for fork in raw['urls']:
                                unpatched_entries.append(UnpatchedFork(
                                                                unpatched_heads=raw["unpatched_heads"],
                                                                url=fork,
                                                                range=range_obj,
                                                                timestamps=raw["timestamps"],
                                                                        ))
                        elif 'url' in raw:
                            unpatched_entries.append(UnpatchedFork(
                                                            unpatched_heads=raw["unpatched_heads"],
                                                            url=raw['url'],
                                                            range=range_obj,
                                                            timestamps=raw["timestamps"],
                                                                    ))
                        else:
                            raise ValueError("No url or urls field in the entry")
                        
                        return unpatched_entries
    def analyze(self) -> "Analysis":
        with pyzstd.open(self.input_path, 'r') as file:
            for line in file:
                        raw = json.loads(line)
                        unpatched_entries=Analysis.obj_to_UnpatchedFork(raw)
                        for unpatched_entry in unpatched_entries:
                            if self.filter_entry(unpatched_entry):
                                    self.result.append(unpatched_entry)
                        
        return self
       

    def filter_entry(self,entry:UnpatchedFork)->bool:
        return True

    
    def print(self, n):
        for entry in self.result[:n]:
            print(asdict(entry))
        return self
    def print_stats(self) -> "Analysis":
         fork_set = set()
         original_set = set()
         vuln_set = set()
         fork_vuln_set = set()
         head_set = set()

         for entry in self.result:
            
            
                original_set.add(entry.range.repo_url)
                fork_set.add(entry.url)
                vuln_set.add(entry.range.vulnerability_id)
                fork_vuln_set.add((entry.url,entry.range.vulnerability_id))
                head_set.update(entry.unpatched_heads)

        
         print ({
            "original_repo":len(original_set),
            "fork":len(fork_set),
            "vuln":len(vuln_set),
            "fork_vuln":len(fork_vuln_set),
            "len_entry":len(self.result),
            "len_head":len(head_set),
         })
         return self
    @staticmethod
    def get_stats_from_file(path):
        fork_set = set()
        original_set = set()
        vuln_set = set()
        fork_vuln_set = set()
        head_set = set()
        with pyzstd.open(path, 'r') as file:
                for line in file:
                        raw = json.loads(line)
                        vulnerability_id=raw["range"].get("vulnerability_id"),
                        repo_url=raw["range"].get("repo_url"),
                        heads=raw["unpatched_heads"]

                        fork_urls = raw['urls'] if 'urls' in raw else [raw['url']] if 'url' in raw else []
                        for fork_url in fork_urls:
                                original_set.add(repo_url)
                                fork_set.add(fork_url)
                                vuln_set.add(vulnerability_id)
                                fork_vuln_set.add((fork_url,vulnerability_id))
                                head_set.update(heads)
        print ({
            "original_repo":len(original_set),
            "fork":len(fork_set),
            "vuln":len(vuln_set),
            "fork_vuln":len(fork_vuln_set),
            "len_head":len(head_set),
         })
                        
                        


    
    def get_stat_object(self) -> dict[str,int]:
         fork_set = set()
         original_set = set()
         vuln_set = set()
         fork_vuln_set = set()
         head_set = set()

         for entry in self.result:
            
            
                original_set.add(entry.range.repo_url)
                fork_set.add(entry.url)
                vuln_set.add(entry.range.vulnerability_id)
                fork_vuln_set.add((entry.url,entry.range.vulnerability_id))
                head_set.update(entry.unpatched_heads)

        
         return ({
            "fork":fork_set,
            "vuln":vuln_set,
            "fork_vuln":fork_vuln_set,
            "head":head_set
         })
    def list_forks(self) -> List[str]:
            fork_set = set()
            for entry in self.result:
                    fork_set.add(entry.url)
            return list(fork_set)

    def export(self) -> "Analysis":
        with pyzstd.open(self.output_path, 'w') as f:
            for item in self.result:
                line = json.dumps(item.to_dict()) + '\n'
                f.write(line.encode('utf-8'))
        return self
    def load_output(self) -> "Analysis":
        def line_generator():
            with pyzstd.open(self.output_path, 'r') as file:
                for line in file:
                    yield line.strip()
        
        with mp.Pool() as pool:
            # Process lines as they're read, without storing all in memory
            for result in pool.imap(Analysis.parse_json_line, line_generator(), chunksize=50):
                self.result.append(result)
        
        print(f"Loaded {len(self.result)} entries from {self.output_path}")
        return self
        
    
    def parse_json_line(line: str) -> UnpatchedFork:
        raw = json.loads(line)
        return Analysis.obj_to_UnpatchedFork(raw)

### STAGE 0 : No filtering, just load and print stats

In [22]:
Analysis.get_stats_from_file(metrics_5_path)

{'original_repo': 3157, 'fork': 1763500, 'vuln': 15117, 'fork_vuln': 53496067, 'len_head': 6627008}


### STAGE 1 : Filter Fork hosted on Github

In [7]:
## Filter repositories not on github 
input = metrics_5_path
metrics_5_gihub_stage = metrics_5_path.replace('.jsonl.zst','_github.jsonl.zst')
class GithubFilterStage(Analysis):
        
     def __init__(self, reposMetadata:ReposMetadata=reposMetadata ,input_path=input, output_path=metrics_5_gihub_stage):
             super().__init__(input_path, output_path)
             self.reposMetadata=reposMetadata

     def filter_entry(self,entry:UnpatchedFork)->bool:                                       
                        # Filter only forks with metadata
                        return entry.url  in self.reposMetadata.reposMaps and (self.reposMetadata.reposMaps.get(entry.url) is not None)
                        

In [ ]:
GithubFilterStage().analyze()\
                   .print_stats()\
                   .export()

In [9]:
Analysis.get_stats_from_file(metrics_5_gihub_stage)

{'original_repo': 3119, 'fork': 1453298, 'vuln': 14399, 'fork_vuln': 40957631, 'len_head': 4030246}


### STAGE 2 : Filter Fork based on their popularity (github stars >100 and github forks >10)

In [10]:
metrics_5_popularity_stage = metrics_5_path.replace('.jsonl.zst','_popularity.jsonl.zst')

class PopularityFilterStage(Analysis):
     def __init__(self, reposMetadata:ReposMetadata=reposMetadata ,input_path=metrics_5_gihub_stage, output_path=metrics_5_popularity_stage):
             super().__init__(input_path, output_path)
             self.reposMetadata=reposMetadata

     def filter_entry(self,entry:UnpatchedFork)->bool:                                       
                        # Filter only forks with metadata
                        repoMetadata : RepoMetadata = self.reposMetadata.reposMaps.get(entry.url)
                        if repoMetadata is not None:
                                return repoMetadata.stars > 100 and repoMetadata.forks > 10
                        else:
                                raise ValueError("No metadata for repo")
                                    

In [11]:
PopularityFilterStage().analyze()\
                   .print_stats()\
                   .export()\
                   

{'original_repo': 633, 'fork': 1364, 'vuln': 5503, 'fork_vuln': 30665, 'len_entry': 1673901, 'len_head': 788131}


In [ ]:
PopularityFilterStage().load_output()\
                   .print_stats()
                   

### STAGE 3 : Filter vulnerability with criticity > 7

In [12]:
metrics_5_criticity_stage = metrics_5_path.replace('.jsonl.zst','_criticity.jsonl.zst')

class CriticityFilterStage(Analysis):
     
     def __init__(self, input_path=metrics_5_popularity_stage, output_path=metrics_5_criticity_stage):
             super().__init__(input_path, output_path)

     def filter_entry(self,entry:UnpatchedFork)->bool:  
        # Consider only entries with cvss

        if entry.range.severity is not None and entry.range.severity != "":                                      
                if not entry.range.severity.startswith("CVSS:3"):  
                      raise ValueError(f"Unknown severity format {entry.range.severity}")   
                else:    
                      return CVSS3(entry.range.severity).base_score >7                         
        return  False
     

In [13]:
CriticityFilterStage().analyze()\
                     .print_stats()\
                     .export()\
                            

{'original_repo': 457, 'fork': 1083, 'vuln': 2247, 'fork_vuln': 11726, 'len_entry': 570420, 'len_head': 583703}


### STAGE 4 : Filter Fork updated after 2023-01-01

In [14]:
metrics_5_timestamp_stage = metrics_5_path.replace('.jsonl.zst','_timestamp.jsonl.zst')

class TimestampFilterStage(Analysis):
     filter_timestamp = datetime(2023,1,1).timestamp()

     def __init__(self, input_path=metrics_5_criticity_stage, output_path=metrics_5_timestamp_stage):
             super().__init__(input_path, output_path)

     def filter_entry(self,entry:UnpatchedFork)->bool:  
        timestamps_filtered=[]
        heads_filtered=[]
        for i in range(len(entry.timestamps)):
            if entry.timestamps[i] > self.filter_timestamp:
                heads_filtered.append(entry.unpatched_heads[i])
                timestamps_filtered.append(entry.timestamps[i])
        
        return len(timestamps_filtered) > 0 

In [15]:
TimestampFilterStage().analyze()\
                     .print_stats()\
                        .export()

{'original_repo': 279, 'fork': 512, 'vuln': 1523, 'fork_vuln': 3664, 'len_entry': 13178, 'len_head': 314330}


### STAGE 5: Filter out archived forks 

In [16]:
metrics_5_archived_stage = metrics_5_path.replace('.jsonl.zst','_archived.jsonl.zst')

class ArchivedFilterStage(Analysis):
     archived_store:RepoMetadataStore = RepoMetadataStore(f"{metadata_store_path}archived.jsonl", workspace,ArchivedMetadata)

     
     def __init__(self, input_path=metrics_5_timestamp_stage, output_path=metrics_5_archived_stage):
             super().__init__(input_path, output_path)


     def filter_entry(self,entry:UnpatchedFork)->bool:
        return self.archived_store.get(entry.url)

                            

Loading JSONL file from ./FINAL_SWH_OSV/data/metadata_store/archived.jsonl
init over


In [17]:
ArchivedFilterStage().analyze()\
                     .print_stats()\
                     .export()

{'original_repo': 268, 'fork': 478, 'vuln': 1478, 'fork_vuln': 3447, 'len_entry': 12186, 'len_head': 289668}


### STAGE 6 : Filter Fork not impacted in their main branch

In [18]:
import logging
def _setup_logger():
    print("Setting up logger")
    log_filename = 'log/main.log'
    os.makedirs(os.path.dirname(log_filename), exist_ok=True)

    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    logger.propagate = False  # prevent duplication

    # Remove any existing handlers
    if logger.hasHandlers():
        logger.handlers.clear()

    # File handler
    file_handler = logging.FileHandler(log_filename)
    file_formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # Console handler
    console_handler = logging.StreamHandler()
    console_formatter = logging.Formatter('[%(levelname)s] %(message)s')
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    logger.info("Logger initialized")
    return logger

setup_logger = _setup_logger()
logging.info("test message")

[INFO] Logger initialized
[INFO] test message


Setting up logger


In [19]:
metrics_5_main_branch_stage = metrics_5_path.replace('.jsonl.zst','_main_branch.jsonl.zst')
swhid_to_branch_path  = f"{metadata_store_path}/new_swhid_to_branch.json"
class MainBranchFilterStage(Analysis):
     not_resolved_head =set()
     skipped_forked = set()
     
     def __init__(self, input_path=metrics_5_archived_stage, output_path=metrics_5_main_branch_stage):
             self.main_branch_store:RepoMetadataStore = RepoMetadataStore(f"{metadata_store_path}main_branch", workspace,MainBranchMetadata)
             super().__init__(input_path, output_path)
             with open(swhid_to_branch_path, "r") as f:
                self.swhid2branch = json.load(f)
                

     def filter_entry(self,entry:UnpatchedFork)->bool:
        
        #Retrieve the main branch of the fork
        main_branch = self.main_branch_store.get(entry.url)

        if main_branch is None or main_branch=="-1":
                #logging.info(f"No main branch for repo {entry.url}")
                return False

        #For each unpatched head, check if it is in the main branch
        filtered_heads=[]

        for head in entry.unpatched_heads:
                swhid_head = head
                found=False
                if swhid_head in self.swhid2branch:
                        current_branches =self.swhid2branch.get(swhid_head)
                        current_branches= [branch.strip().split('/')[-1] for branch in current_branches]
                        if  main_branch in current_branches:
                                filtered_heads.append(head)
                        found=True
                else:
                        self.not_resolved_head.add(head)
        if len(filtered_heads) ==0 and not found:
                    self.skipped_forked.add(entry.url)
        entry.unpatched_heads=filtered_heads
        return len(filtered_heads)>0
                             

     def get_swhid_head(self,head):
        return f"swh:1:rev:{head}"

     def export_not_resolved(self):
        print(f"{analysis_out_path}/not_resolved_head.json")
        with open(f"{analysis_out_path}/not_resolved_head.json", 'w') as f:
            json.dump(list(self.not_resolved_head), f, indent=4)
        return self

In [20]:
main_branch_filter_stage=MainBranchFilterStage()
main_branch_filter_stage.analyze()\
                     .print_stats()\
                        .export()\
                    .export_not_resolved()  

print(f"Not resolved heads number : {len(main_branch_filter_stage.not_resolved_head)}")
print(f"Skipped forked repos number due to missing head : {len(main_branch_filter_stage.skipped_forked)}")

Loading JSONL file from ./FINAL_SWH_OSV/data/metadata_store/main_branch
init over
{'original_repo': 132, 'fork': 188, 'vuln': 705, 'fork_vuln': 1130, 'len_entry': 1556, 'len_head': 543}
./FINAL_SWH_OSV/data//analysis//not_resolved_head.json
Not resolved heads number : 809
Skipped forked repos number due to missing head : 7


## Stage 7 - Sibling

In [21]:

import pickle
with open("range.pkl", "rb") as f:
    range_list = pickle.load(f)
len(range_list)

metrics_5_sibling = metrics_5_path.replace('.jsonl.zst','sibling.jsonl.zst')
class SiblingFilterStage(Analysis):
    

     
     def __init__(self,original_range_list, input_path=metrics_5_main_branch_stage, output_path=metrics_5_sibling,):
           
             super().__init__(input_path, output_path)
             self.original_range_list=original_range_list
             


        
     def filter_entry(self,entry:UnpatchedFork)->bool:
        upstreams_associated_to_range= self.get_upstream_repo_having_fix(entry.range.vulnerability_id)
        
        if  upstreams_associated_to_range and entry.url in upstreams_associated_to_range and len(upstreams_associated_to_range) > 1:
                print(f"Sibling range found for {entry.url} {entry.range.vulnerability_id}")
                return 0
      
        return 1
            
     def get_upstream_repo_having_fix(self,cve_id):
        result=set()
        for range in self.original_range_list:
            if range.vulnerability_id == cve_id and (EventType.FIXED in range.events.values() or EventType.LAST_AFFECTED in range.events.values()):
                result.add(range.repo_url)
        return result

SiblingFilterStage(range_list).analyze()\
                     .print_stats()\
                     .export()


Sibling range found for https://github.com/torproject/tor CVE-2021-28089
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-36092
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-36092
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-36092
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-36090
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-36090
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-36090
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2023-29213
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2023-29213
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2023-29213
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-23619
Sibling range found for https://github.com/xwiki/xwiki-platform CVE-2022-23619
Sibling range found for https://github.com/xwiki/xwiki-pla

### STAGE 8 : Filter (Fork,Vuln) where fork do not diverge

In [22]:
metrics_5_main_branch_stage = metrics_5_path.replace('.jsonl.zst','_main_branch.jsonl.zst')
workspace = "/tmp/workspace"
metrics_5_divergence_stage = metrics_5_path.replace('.jsonl.zst','divergence.jsonl.zst')

In [23]:
class DivergenceFilterStage(Analysis):
     divergence_store:HeadMetadataStore = HeadMetadataStore(f"{metadata_store_path}divergence.jsonl", workspace,DivergenceMetadata,read_only=False)

     
     def __init__(self, input_path=metrics_5_sibling, output_path=metrics_5_divergence_stage):
             super().__init__(input_path, output_path)
             
     def filter_entry(self,entry:UnpatchedFork)->bool:
        filtered_heads=[]
        for head in entry.unpatched_heads:
                not_divergent= self.divergence_store.get(entry.url,entry.range,head)==1
                if not_divergent:
                        filtered_heads.append(head)
                        
        entry.unpatched_heads=filtered_heads    
        return len(filtered_heads)>0
            
        
DivergenceFilterStage().analyze()\
                     .print_stats()\
                        .export()

Initializing DivergenceMetadata metadata store
Loading JSONL file from ./FINAL_SWH_OSV/data/metadata_store/divergence.jsonl
{'original_repo': 55, 'fork': 86, 'vuln': 347, 'fork_vuln': 586, 'len_entry': 703, 'len_head': 330}


## Filter data quality 

In [ ]:
import pandas as pd

data_quality_df = pd.read_csv(manual_data_quality)
data_quality_df

In [25]:
metrics_5_data_quality_stage = metrics_5_path.replace('.jsonl.zst','data_quality.jsonl.zst')
class DataQualityFilterStage(Analysis):

     
     def __init__(self, input_path=metrics_5_divergence_stage, output_path=metrics_5_data_quality_stage):
             super().__init__(input_path, output_path)
             self.data_quality_df = pd.read_csv(manual_data_quality)
             
     def filter_entry(self, entry: UnpatchedFork) -> bool:
        vuln = entry.range.vulnerability_id
        upstream_url = entry.range.repo_url

        # Select the matching rows
        row = data_quality_df[
            (data_quality_df['vuln_id'].astype(str) == str(vuln)) &
            (data_quality_df['upstream'].astype(str) == str(upstream_url))
        ]

        if row.empty:
            raise ValueError(f"No data quality entry for vuln={vuln}, upstream={upstream_url}")
           
        if len(row) > 1:
            raise ValueError(f"Multiple data quality entries for vuln={vuln}, upstream={upstream_url}")

        # Extract the scalar value safely
        is_valid = bool(row['Valid'].iloc[0])
        return is_valid
            

f=DataQualityFilterStage().analyze()\
                     .print_stats()\
                        .export()

{'original_repo': 31, 'fork': 49, 'vuln': 204, 'fork_vuln': 330, 'len_entry': 381, 'len_head': 271}


In [26]:
metrics_5_cherry_pick_stage = metrics_5_path.replace('.jsonl.zst','cherry_pick.jsonl.zst')
class CherryPickFilterStage(Analysis):

     
     def __init__(self, input_path=metrics_5_data_quality_stage, output_path=metrics_5_cherry_pick_stage):
             super().__init__(input_path, output_path)
             self.cherry_pick_store = RangeMetadataStore(f"{metadata_store_path}cherry_pick.jsonl", workspace,CherryPickMetadata,read_only=False)

     def filter_entry(self,entry:UnpatchedFork)->bool:
         
        return self.cherry_pick_store.get(entry.url,entry.range) == 1

stat_obj=CherryPickFilterStage().analyze()\
                     .print_stats()\
                     .export()\
                     .export_as_csv("FINAL_SWH_OSV/manual_analysis/out.csv").get_stat_object()

Loading JSONL file from ./FINAL_SWH_OSV/data/metadata_store/cherry_pick.jsonl
{'original_repo': 26, 'fork': 41, 'vuln': 152, 'fork_vuln': 195, 'len_entry': 212, 'len_head': 238}
